# SongForge-DL — Remote Colab Runner

Runs the project on Colab GPU + Google Drive. Nothing heavy touches the local workstation.

**Run the cells in order, top to bottom.** Each cell is a milestone gate and stops loudly if it fails.

| Cell | Stage | Gate |
| --- | --- | --- |
| 1 | Config | — |
| 2 | Mount Drive + GPU check | GPU present |
| 3 | Get the code onto Colab | project dir found |
| 4 | Install + M00 + M01 | `pytest -q` passes, registry valid |
| 5 | Dataset (needs terms acceptance) | audio present in Drive |
| 6 | **M02 audio preprocessing** | manifests written, no split leakage |
| 7 | M03 codec spike (not a milestone gate) | loss falls, RVQ not collapsed |
| 8 | Persist logs to Drive | logs copied |

## Milestone status

M00 PASS, M01 PASS, **M02 is the milestone under test**. M03 codec code exists and runs,
but it was built before M02 and is kept as an **experimental spike** — cell 7 does not mark
M03 accepted. M04+ has not started.

## Before you start

`SETUP_MODE = "zip"` is the default and needs **no GitHub account**.
Upload `songforge-colab.zip` to your Drive at `MyDrive/songforge-dl/` first.

## 1. Config — edit this cell, then run everything below

In [ ]:
# ---- how the code gets onto Colab -------------------------------------
# "zip" = upload songforge-colab.zip to Drive (no GitHub needed)  <-- default
# "git" = clone from GitHub (only if the repo actually exists)
SETUP_MODE = "zip"

DRIVE_ROOT = "/content/drive/MyDrive/songforge-dl"
ZIP_PATH   = f"{DRIVE_ROOT}/songforge-colab.zip"   # upload the archive here
WORK_DIR   = "/content/songforge"                  # local Colab disk: fast. Drive is slow.

# ---- only used when SETUP_MODE == "git" --------------------------------
REPO_URL     = "https://github.com/auth889-ai/ml-sing.git"
BRANCH       = "main"
GITHUB_TOKEN = ""   # fine-grained token if the repo is private; clear cell output afterwards

PROJECT_SUBDIR = "songforge-dl-starter"
SONGFORGE_DATA = f"{DRIVE_ROOT}/data"

# ---- dataset -----------------------------------------------------------
ACCEPT_NONCOMMERCIAL_DATASET_TERMS = False   # set True only after reading the upstream licence
DATASET_ID = "babyslakh"                     # babyslakh, slakh2100, lakh_midi, gtsinger, mtg_jamendo, nsynth

# ---- M02 audio preprocessing (the milestone under test) ----------------
M02_RAW_DIR     = f"{SONGFORGE_DATA}/raw/babyslakh"
M02_OUTPUT_DIR  = f"{SONGFORGE_DATA}/processed/babyslakh_m02"
M02_LIMIT_FILES = 24       # keep the acceptance run tiny; None uses everything
M02_SPLIT_MODE  = "song"   # song = song-disjoint, singer = singer-disjoint
M02_SEED        = 42

# ---- M03 codec spike (NOT an accepted milestone) -----------------------
M03_AUDIO_GLOB = f"{SONGFORGE_DATA}/raw/babyslakh/**/*.wav"
M03_OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/codec_m03_spike"
M03_STEPS      = 600   # NOT 80. The RVQ codebook needs warm-up or the collapse gate fails.

print("config loaded")

## 2. Mount Drive and confirm the GPU

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

for sub in ["", "/data/raw", "/data/processed", "/data/manifests",
            "/checkpoints", "/logs", "/outputs"]:
    os.makedirs(DRIVE_ROOT + sub, exist_ok=True)
print("Drive ready:", DRIVE_ROOT)

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime > Change runtime type > Hardware accelerator: GPU, then rerun.")

## 3. Get the code onto Colab

This is the cell that used to fail. It now stops with a clear message instead of
letting later cells run from the wrong directory.

In [ ]:
import os
import shutil
import subprocess
import zipfile
from urllib.parse import urlparse


def fail(message):
    raise SystemExit("SETUP FAILED\n" + message)


if SETUP_MODE == "zip":
    if not os.path.exists(ZIP_PATH):
        fail(
            f"{ZIP_PATH} not found.\n\n"
            "Fix: upload songforge-colab.zip from your Mac to Google Drive at\n"
            f"  MyDrive/songforge-dl/\n"
            "The archive is at ~/connect-your-learning/ml-sing/songforge-colab.zip"
        )
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(WORK_DIR)
    project_dir = os.path.join(WORK_DIR, PROJECT_SUBDIR)

elif SETUP_MODE == "git":
    repo_dir = os.path.join(WORK_DIR, "repo")
    clone_url = REPO_URL
    if GITHUB_TOKEN:
        parsed = urlparse(REPO_URL)
        if parsed.scheme != "https":
            fail("GITHUB_TOKEN auth expects an https GitHub URL")
        clone_url = f"https://x-access-token:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}"
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    os.makedirs(WORK_DIR, exist_ok=True)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, repo_dir], text=True
    )
    if result.returncode != 0:
        fail(
            "git clone failed.\n"
            "The repository does not exist, is private, or the branch is wrong.\n"
            "Fix: set GITHUB_TOKEN in cell 1, or switch to SETUP_MODE = 'zip'."
        )
    project_dir = os.path.join(repo_dir, PROJECT_SUBDIR)

else:
    fail(f"SETUP_MODE must be 'zip' or 'git', got {SETUP_MODE!r}")

if not os.path.isfile(os.path.join(project_dir, "pyproject.toml")):
    fail(f"pyproject.toml not found in {project_dir}. Check PROJECT_SUBDIR.")

os.chdir(project_dir)
os.environ["SONGFORGE_DATA"] = SONGFORGE_DATA

missing = [
    name
    for name in ("scripts/colab_m03_acceptance.py", "scripts/train_codec.py",
                 "configs/codec/codec_m03_tiny.yaml", "src/songforge/models/codec/quantizer.py")
    if not os.path.exists(name)
]
if missing:
    fail("This build is missing M03 files: " + ", ".join(missing) + "\nRe-export songforge-colab.zip.")

print("project dir :", os.getcwd())
print("M03 files   : present")

## 4. Install + M00 (tests) + M01 (dataset registry)

In [ ]:
import subprocess

!pip install -q -e '.[dev,audio]'

print("\n=== M00: test suite ===")
m00 = subprocess.run(["python", "-m", "pytest", "-q"], text=True)

print("\n=== M01: dataset registry ===")
m01 = subprocess.run(["python", "scripts/validate_dataset_registry.py"], text=True)

if m00.returncode != 0:
    raise SystemExit("M00 FAILED: tests did not pass. Do not continue.")
if m01.returncode != 0:
    raise SystemExit("M01 FAILED: dataset registry invalid. Do not continue.")
print("\nM00 PASS and M01 PASS")

## 5. Dataset

Read the upstream licence first. Gated/non-commercial sets only download after you set
`ACCEPT_NONCOMMERCIAL_DATASET_TERMS = True` in cell 1.

In [ ]:
import subprocess

import yaml

with open("configs/data/datasets.yaml", "r", encoding="utf-8") as f:
    registry = yaml.safe_load(f)

spec = registry["datasets"][DATASET_ID]
needs_acceptance = spec["license"].get("requires_user_acceptance") or spec["access"].get("gated")

print(f"Selected : {DATASET_ID} - {spec['name']}")
print(f"License  : {spec['license']['name']}")
print(f"Size     : {spec['access'].get('estimated_size')}")

if needs_acceptance and not ACCEPT_NONCOMMERCIAL_DATASET_TERMS:
    raise SystemExit(
        "This dataset requires you to accept upstream terms.\n"
        "Read the licence, then set ACCEPT_NONCOMMERCIAL_DATASET_TERMS = True in cell 1."
    )

for command in spec["access"].get("colab_commands", []):
    print("RUN:", command)
    subprocess.run(command, shell=True, check=True)

import glob
found = glob.glob(M03_AUDIO_GLOB, recursive=True)
print(f"\naudio files matching M03_AUDIO_GLOB: {len(found)}")
if not found:
    print("No audio yet. M03 needs real audio at:", M03_AUDIO_GLOB)

## 6. M02 — audio preprocessing and dataset infrastructure

**This is the milestone under test.** It takes the approved BabySlakh subset through:

    raw audio -> validation -> preprocessing -> segmentation
              -> canonical manifest -> train/val/test split -> Drive persistence

Corrupt, empty, and silent audio is rejected rather than trained on. Splits are assigned to
whole songs, so no segment of a song can appear in two splits. Everything lands on Drive.

In [ ]:
import subprocess

limit = ["--limit-files", str(M02_LIMIT_FILES)] if M02_LIMIT_FILES else []
m02 = subprocess.run(
    ["python", "scripts/colab_m02_acceptance.py",
     "--dataset-id", "babyslakh",
     "--audio-dir", M02_RAW_DIR,
     "--output-dir", M02_OUTPUT_DIR,
     "--split-mode", M02_SPLIT_MODE,
     "--seed", str(M02_SEED)] + limit,
    text=True,
)
if m02.returncode != 0:
    raise SystemExit(
        "M02 acceptance FAILED. Read the report above.\n"
        f"Details: {M02_OUTPUT_DIR}/m02_acceptance_report.json"
    )
print("\nM02 acceptance PASS")

### M02 verification — manifests, splits, and leakage

Reads back what was written to Drive and checks it independently of the runner.

In [ ]:
import json
import sys

sys.path.insert(0, "src")

from songforge.data.dedup import assert_no_cross_split_duplicates, duplicate_report
from songforge.data.manifest import (
    assert_no_track_leakage,
    assert_provenance_complete,
    assert_singer_disjoint,
    manifest_summary,
    read_jsonl,
)
from songforge.data.splits import assert_group_disjoint, split_report

records = read_jsonl(f"{M02_OUTPUT_DIR}/manifests/all.jsonl")
print(f"records read back from Drive: {len(records)}")

assert_no_track_leakage(records)
assert_group_disjoint(records, M02_SPLIT_MODE)
assert_singer_disjoint(records)
assert_no_cross_split_duplicates(records)
assert_provenance_complete(records)
print("no song leakage, no singer leakage, no cross-split duplicate audio, provenance intact")

print("\n-- manifest summary --")
print(json.dumps(manifest_summary(records), indent=2, sort_keys=True))
print("\n-- split report --")
print(json.dumps(split_report(records, M02_SPLIT_MODE), indent=2, sort_keys=True))
print("\n-- duplicates --")
print(json.dumps(duplicate_report(records), indent=2, sort_keys=True))

example = records[0]
print("\n-- one canonical record --")
print(json.dumps(example.to_dict(), indent=2, sort_keys=True))

!ls -lh "$M02_OUTPUT_DIR/manifests"

## 7. M03 codec spike — NOT a milestone gate

M03 was implemented before M02 and is preserved as an **experimental spike**. Running this
cell does not mark M03 accepted; official M03 acceptance comes after M02 is signed off.

It needs a CUDA runtime. `M03_STEPS = 600` is deliberate: the residual codebook starts
collapsed and needs a few hundred steps to spread out, so at 80 steps the run fails its own
`rvq_collapse_suspected` gate even while training correctly.

The codec can now read the M02 manifest directly with `--manifest`, which is the path future
milestones should use instead of a raw glob.

In [ ]:
# Trains from the M02 manifest produced above, not a raw glob.
!python scripts/colab_m03_acceptance.py \
    --config configs/codec/codec_m03_tiny.yaml \
    --audio-glob "$M03_AUDIO_GLOB" \
    --output-dir "$M03_OUTPUT_DIR" \
    --steps $M03_STEPS

!ls -lh "$M03_OUTPUT_DIR"

## 8. Persist logs to Drive

In [ ]:
import json
import os
import shutil

for name in ("EXPERIMENT_LOG_M02.md", "EXPERIMENT_LOG.md"):
    source = os.path.join("docs", name)
    if os.path.exists(source):
        shutil.copy(source, f"{DRIVE_ROOT}/{name}")
        print("copied ->", f"{DRIVE_ROOT}/{name}")
    else:
        print("not produced yet:", source)

m02_report = os.path.join(M02_OUTPUT_DIR, "m02_acceptance_report.json")
if os.path.exists(m02_report):
    report = json.load(open(m02_report))
    print("\n== M02 ==")
    print("acceptance_pass :", report["acceptance_pass"])
    print("segments        :", report["manifest"]["segments"])
    print("songs           :", report["manifest"]["tracks"])
    print("leakage         :", report["leakage"])

m03_report = os.path.join(M03_OUTPUT_DIR, "m03_acceptance_report.json")
if os.path.exists(m03_report):
    acceptance = json.load(open(m03_report))["acceptance"]
    print("\n== M03 spike (not an accepted milestone) ==")
    print("loss", acceptance["first_loss"], "->", acceptance["final_loss"])
    print("rvq_collapse_suspected:", acceptance["rvq_collapse_suspected"])

print("\nM02 is the milestone under test. M03 remains an experimental spike; M04+ not started.")